In [ ]:
import os
import cv2
import json
import base64
from pathlib import Path
from ultralytics import YOLO
from ultralytics.utils.files import increment_path
import shutil

model = YOLO(r'/App/lpl/yolo11/runs/train/exp34/weights/best.pt')   # 加载模型
image_dir = r"/App/lpl/yolo11/FY_4_project/datasets/images/test"
images = os.listdir(image_dir)                                      # 所有待检测图片
print("total images number:", len(images))

base_datasets = Path(r"/App/lpl/yolo11/Auto_label") / "Datasets"
datasets_dir = increment_path(base_datasets, exist_ok=False, mkdir=True)   # 自动递增

output_json_dir   = datasets_dir / "json"
output_image_dir  = datasets_dir / "image"

output_json_dir.mkdir(parents=True, exist_ok=True)
output_image_dir.mkdir(parents=True, exist_ok=True)

# 若模型自带类别名称（推荐），直接使用；否则自行定义
class_names = model.names if hasattr(model, "names") else ["class0", "class1"]
print('class_names', class_names)

for i, image_name in enumerate(images):
    image_path = os.path.join(image_dir, image_name) # 待标注的图像文件
    print(f"Processing {i+1}/{len(images)}: {image_name}")

    # 直接使用图片路径进行预测
    results = model.predict(
        source=image_path,
        imgsz=1088,
        conf=0.45,
        iou=0.45,
        device=0,
        half=False,
        save=False,          # 保存检测后图片
        save_txt=False,      # 保存 txt 标签
        save_conf=True,     # txt 中写入置信度
        project=output_json_dir,
        name=f"batch_{i//10}",
        exist_ok=True
    )

    # 读取原图尺寸（用于 JSON）
    img = cv2.imread(image_path)
    h, w = img.shape[:2]

    # 生成并保存 LabelMe JSON
    for result in results:
        json_dict = {
            "version": "2.3.6",          # 可自行修改为当前 LabelMe 版本
            "flags": {},
            "shapes": [],
            "imagePath": image_name,
            "imageData": None,             # 留空，LabelMe 会自行读取本地文件
            "imageHeight": h,
            "imageWidth": w
        }

        # 遍历每个检测框
        boxes = result.boxes
        if boxes is not None and len(boxes):
            xyxy = boxes.xyxy.cpu().numpy()      # (N,4)  左上x,左上y,右下x,右下y
            confs = boxes.conf.cpu().numpy()    # (N,)
            cls_ids = boxes.cls.cpu().numpy().astype(int)  # (N,)

            for bbox, conf, cls_id in zip(xyxy, confs, cls_ids):
                x1, y1, x2, y2 = map(float, bbox)
                label_name = class_names[cls_id] if cls_id < len(class_names) else f"class_{cls_id}"
                label = f"{label_name}"

                shape = {
                    "label": label,
                    # "points": [[x1, y1], [x2, y2]],
                    "points": [
                                [float(x1), float(y1)],   # 左上
                                [float(x2), float(y1)],   # 右上
                                [float(x2), float(y2)],   # 右下
                                [float(x1), float(y2)]    # 左下
                            ],
                    "group_id": None,
                    "description": "",
                    "difficult": False,
                    "shape_type": "rectangle",
                    "flags": {},
                    "attributes": {}
                }
                json_dict["shapes"].append(shape)
        
        # 保存图片文件
        shutil.copy(image_path, output_image_dir)
        print(f"图片 已保存: {output_image_dir}")

        # 保存 JSON 文件
        json_name = os.path.splitext(image_name)[0] + ".json"
        json_path = os.path.join(output_json_dir, json_name)
        with open(json_path, "w", encoding="utf-8") as f:
            json.dump(json_dict, f, ensure_ascii=False, indent=2)

        print(f"LabelMe JSON 已保存: {json_path}")


print(f"所有图片处理完成！图片文件保存在：{output_image_dir}，json 文件保存在: {output_json_dir}")